# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` as specified in the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset @id: {metadata.id}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, each table or data source is a RecordSet with a unique `@id`. Fields and columns are also referenced by their `@id`.

Let's view all available RecordSets, their fields, and column IDs.

In [ ]:
# List RecordSets and their details
record_sets = dataset.metadata.record_sets
print("RecordSets available in the dataset:")
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields (columns):")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All RecordSets and fields are referenced by their `@id`.

Below, we will extract all available RecordSets, storing each in a DataFrame. Replace `<record_set_id>` as needed for exploration.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for RecordSet {record_set_id}: {df.columns.tolist()}")
    print(f"Sample records for {record_set_id}:")
    print(df.head(), "\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

For demonstration, we select a numeric field from one RecordSet, filter values, normalize, and optionally group.

In [ ]:
# Choose a RecordSet and fields for EDA.
# Replace these IDs below with those shown in the previous step, e.g. from hormone levels table.
example_record_set_id = record_set_ids[1] if len(record_set_ids) > 1 else record_set_ids[0]
df = dataframes[example_record_set_id]

# Find a numeric field in this RecordSet
fields = [field for field in record_sets if field.id == example_record_set_id][0].fields

# Choose the first numeric field
numeric_field_id = None
for field in fields:
    if field.data_type in ["Float", "Integer", "Number"]:
        numeric_field_id = field.id
        break

if numeric_field_id is not None:
    print(f"Using numeric field: {numeric_field_id}")
    try:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != 'O' else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # If a categorical field exists, group
        group_field_id = None
        for field in fields:
            if field.data_type in ["Text", "Boolean"] and field.id != numeric_field_id:
                group_field_id = field.id
                break

        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    except Exception as e:
        print(f"Error processing EDA: {e}")
else:
    print("No numeric field found for EDA in this RecordSet.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is a simple histogram visualization of a numeric field, using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric values from the filtered DataFrame
if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {example_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrates how to load, inspect, and process a FAIR^2 dataset defined by a Croissant schema using the `mlcroissant` library. All data entities were referenced by their `@id` for schema fidelity.

Key steps included:
- Loading structured metadata and tabulations for clinical, hormone, imaging, and genetic data.
- Examining RecordSets and their fields.
- Extracting and processing records programmatically by their `@id`.
- Performing basic EDA and visualizing data distributions.

For future analyses, explore more advanced statistical methods or link tables via `@id` relationships for richer outcome analyses.